# 📋 CROSS-CHECK FORM KEIKUTSERTAAN SERTIFIKASI
## Verifikasi Status Sertifikasi Peserta

---
**Deskripsi Program:**
- Memproses data dari Form Keikutsertaan Sertifikasi Kompetensi
- Cross-check dengan database Certiport untuk verifikasi status sertifikasi
- **CEK KEDUANYA**: Akan mengecek apakah peserta sudah ikut MOS, MCF, atau keduanya
- Menghasilkan daftar peserta dengan keterangan detail:
  - ✅ **Sudah sertifikasi MOS & MCF (keduanya)**
  - ✅ **Sudah sertifikasi MOS saja (belum MCF)**
  - ✅ **Sudah sertifikasi MCF saja (belum MOS)**
  - ❌ **Belum pernah sertifikasi MOS maupun MCF**

**Input Files:**
1. `FORM KEIKUTSERTAAN SERTIFIKASI KOMPETENSI.csv` - Data form keikutsertaan
2. `certiport.csv` - Database hasil ujian dari Certiport

**Output Files:**
- `FORM_SUDAH_SERTIFIKASI.csv` - Peserta yang sudah pernah sertifikasi (dengan detail keterangan)
- `FORM_BELUM_SERTIFIKASI.csv` - Peserta yang belum pernah sertifikasi
- `HASIL_FORM_KEIKUTSERTAAN.xlsx` - File Excel lengkap dengan multiple sheets

---

## 1️⃣ Import Library & Setup

In [147]:
# ============================================================
# IMPORT LIBRARY YANG DIPERLUKAN
# ============================================================
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import warnings
warnings.filterwarnings('ignore')

print("✅ Library berhasil diimport!")
print("   - pandas: untuk manipulasi data")
print("   - fuzzywuzzy: untuk fuzzy string matching")

✅ Library berhasil diimport!
   - pandas: untuk manipulasi data
   - fuzzywuzzy: untuk fuzzy string matching


## 2️⃣ Load Data Form Keikutsertaan & Certiport

In [148]:
# ============================================================
# LOAD DATA FORM KEIKUTSERTAAN
# ============================================================
print(f"{'='*70}")
print("📋 LOAD DATA FORM KEIKUTSERTAAN SERTIFIKASI")
print(f"{'='*70}")

# Load data form keikutsertaan (dari folder parent)
form_df = pd.read_csv('../FORM KEIKUTSERTAAN SERTIFIKASI KOMPETENSI.csv')

print(f"\n✅ Data Form Keikutsertaan berhasil diload!")
print(f"   Total peserta: {len(form_df)}")

# Tampilkan kolom yang tersedia
print(f"\n📋 Kolom yang tersedia:")
for i, col in enumerate(form_df.columns, 1):
    print(f"   {i:>2}. {col}")

📋 LOAD DATA FORM KEIKUTSERTAAN SERTIFIKASI

✅ Data Form Keikutsertaan berhasil diload!
   Total peserta: 159

📋 Kolom yang tersedia:
    1. ID
    2. Start time
    3. Completion time
    4. Email
    5. Name
    6. Last modified time
    7. NIM (Nomor Induk Mahasiswa)
    8. NAMA LENGKAP
    9. NOMOR HANDHONE
   10. Email2
   11. PROGRAM STUDI
   12. PILIH PROGRAM SERTIFIKASI YANG SUDAH PERNAH DIAMBIL
   13. PILIH PROGRAM SERTIFIKASI YANG SUDAH PERNAH DIAMBIL2
   14. PILIH SUBPROGRAM MOS
   15. PILIH SUBPROGRAM MCF
   16. PILIH SUBPROGRAM
   17. Dengan ini saya menyatakan bahwa data yang saya masukkan adalah BENAR, yang dapat DIPERTANGGUNG JAWABKAN dan jika saya melanggar ketentuan, maka saya bersedia mendapatkan SANKSI sesuai aturan dari...
   18. Catatan: Surat Keterangan Bukti Peserta Sertifikasi diberikan setelah ITCC melakukan validasi data peserta dan akan diterbitkan secepatnya, dan dikirim ke email institusi/email kampus IT-PLN peserta.



In [149]:
# ============================================================
# LOAD DATA CERTIPORT
# ============================================================
print(f"{'='*70}")
print("📊 LOAD DATA CERTIPORT")
print(f"{'='*70}")

# Load data Certiport (dari folder parent)
certiport_df = pd.read_csv('../certiport.csv', skiprows=4)
certiport_df = certiport_df.dropna(axis=1, how='all')

# Filter berdasarkan jenis ujian
certiport_mcf = certiport_df[certiport_df['Exam'].str.contains('AI-900', na=False)].copy()
certiport_mos = certiport_df[certiport_df['Exam'].str.contains('Office 2019', na=False)].copy()

print(f"\n✅ Data Certiport berhasil diload!")
print(f"   Total record    : {len(certiport_df)}")
print(f"   MCF (AI-900)    : {len(certiport_mcf)}")
print(f"   MOS (Office 2019): {len(certiport_mos)}")

# Breakdown Exam
print(f"\n📊 Breakdown Exam di Certiport:")
print(certiport_df['Exam'].value_counts().to_string())

📊 LOAD DATA CERTIPORT

✅ Data Certiport berhasil diload!
   Total record    : 4853
   MCF (AI-900)    : 897
   MOS (Office 2019): 3445

📊 Breakdown Exam di Certiport:
Exam
Microsoft Word (Office 2019)                                         2714
AI-900: Microsoft Azure AI Fundamentals                               897
Microsoft Excel (Office 2019)                                         624
Microsoft Word (Office 2016)                                          287
Microsoft Excel (Office 2016)                                         161
Microsoft PowerPoint (Office 2019)                                    104
Microsoft PowerPoint (Office 2016)                                     53
Microsoft Word Expert (Office 2019)                                     3
Microsoft Excel (Microsoft 365 Apps)                                    2
Microsoft Word (Microsoft 365 Apps)                                     1
SC-900: Microsoft Security, Compliance, and Identity Fundamentals       1


## 3️⃣ Siapkan & Bersihkan Data Form

In [150]:
# ============================================================
# SIAPKAN DAN BERSIHKAN DATA FORM
# ============================================================
print(f"{'='*70}")
print("🔧 PERSIAPAN DATA FORM KEIKUTSERTAAN")
print(f"{'='*70}")

# Bersihkan nama dan NIM
form_df['NAMA LENGKAP'] = form_df['NAMA LENGKAP'].fillna('').str.strip().str.upper()
form_df['NIM (Nomor Induk Mahasiswa)'] = form_df['NIM (Nomor Induk Mahasiswa)'].astype(str).str.strip()

# Buat kolom Email Final (prioritas Email, jika kosong/anonymous gunakan Email2)
def get_email_final(row):
    email = str(row['Email']) if pd.notna(row['Email']) else ''
    email2 = str(row['Email2']) if pd.notna(row['Email2']) else ''
    
    if email and email != 'anonymous' and email.strip() != '' and email.strip().lower() != 'nan':
        return email.strip()
    elif email2 and email2.strip() != '' and email2.strip().lower() != 'nan':
        return email2.strip()
    else:
        return '-'

form_df['Email Final'] = form_df.apply(get_email_final, axis=1)

# Tampilkan preview data
print(f"\n✅ Data berhasil diproses!")
print(f"\n📋 Preview Data (5 baris pertama):")
preview_cols = ['NAMA LENGKAP', 'NIM (Nomor Induk Mahasiswa)', 'Email Final']
# Tambahkan kolom subprogram jika ada
for col in ['PILIH SUBPROGRAM MOS', 'PILIH SUBPROGRAM MCF', 'PILIH SUBPROGRAM']:
    if col in form_df.columns:
        preview_cols.append(col)
print(form_df[preview_cols].head().to_string(index=False))

🔧 PERSIAPAN DATA FORM KEIKUTSERTAAN

✅ Data berhasil diproses!

📋 Preview Data (5 baris pertama):
          NAMA LENGKAP NIM (Nomor Induk Mahasiswa)                   Email Final   PILIH SUBPROGRAM MOS PILIH SUBPROGRAM MCF PILIH SUBPROGRAM
    RATU ADISYA FAMELA                   202231105       ratu2231105@itpln.ac.id  MOS: Office Word 2019                  NaN              NaN
MUHAMMAD RAIHAN AZZAKY                   202241017     raihan2241017@itpln.ac.id MOS: Office Excel 2019                  NaN              NaN
     VIA ISNATUL LAILA                   202241013 viaisnatul2241013@itpln.ac.id  MOS: Office Word 2019                  NaN              NaN
  NAUFAL YURFANA AISAL                   202241012     naufal2241012@itpln.ac.id  MOS: Office Word 2019                  NaN              NaN
  WIDY SYAFITRI SLAMET                   202231070       Widy2231070@itpln.ac.id  MOS: Office Word 2019                  NaN              NaN


## 4️⃣ Fungsi Fuzzy Matching

In [ ]:
# ============================================================
# FUNGSI-FUNGSI UNTUK FUZZY MATCHING (IMPROVED v2)
# ============================================================

def normalize_name(name):
    """Normalisasi nama: uppercase, hapus karakter khusus kecuali - dan ., rapikan spasi"""
    if pd.isna(name):
        return ""
    # Keep alphanumeric, spaces, dash, and dot
    cleaned = ''.join(c for c in str(name) if c.isalnum() or c.isspace() or c in '.-')
    return ' '.join(cleaned.upper().strip().split())

def reverse_name(name):
    """Balik urutan nama (untuk antisipasi nama terbalik)"""
    parts = name.split()
    if len(parts) >= 2:
        return ' '.join(parts[::-1])
    return name

def count_matching_words(name1, name2, min_word_length=2):
    """Hitung jumlah kata yang sama antara dua nama"""
    words1 = set(w for w in name1.split() if len(w) >= min_word_length)
    words2 = set(w for w in name2.split() if len(w) >= min_word_length)
    return len(words1.intersection(words2)), len(words1), len(words2)

def get_min_matching_words(peserta_word_count, certiport_word_count):
    """
    Tentukan minimal kata yang harus cocok dengan aturan RELAXED:
    
    1. Jika Certiport punya ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)
    2. Gunakan persentase 50% dari jumlah kata yang LEBIH SEDIKIT
    
    Contoh:
    - Form: 4 kata, Certiport: 2 kata → min(2, ceil(2*50%)) = min(2, 1) → 1 kata (tapi minimal 2)
    - Form: 4 kata, Certiport: 4 kata → ceil(4*50%) = 2 kata
    - Form: 3 kata, Certiport: 3 kata → ceil(3*50%) = 2 kata
    """
    import math
    
    # Ambil jumlah kata yang lebih sedikit
    min_words = min(peserta_word_count, certiport_word_count)
    
    # SPECIAL RULE: Jika Certiport hanya punya ≤2 kata, relax rule
    if certiport_word_count <= 2:
        # Minimal 2 kata cocok, atau semua kata certiport jika kurang dari 2
        return min(2, certiport_word_count)
    
    # Gunakan 50% dari kata yang lebih sedikit (minimal 2)
    percentage_required = math.ceil(min_words * 0.5)
    return max(2, percentage_required)

def is_single_word_match(name1, name2):
    """
    Untuk nama 1 kata, cek apakah ada tanda - atau . yang menunjukkan kecocokan
    Contoh: 'ABDUL-RAHMAN' atau 'A.RAHMAN'
    """
    # Cek apakah nama mengandung - atau .
    has_special_char = '-' in name1 or '.' in name1 or '-' in name2 or '.' in name2
    if not has_special_char:
        return False
    
    # Normalize tanpa - dan . untuk perbandingan
    clean1 = name1.replace('-', ' ').replace('.', ' ').upper()
    clean2 = name2.replace('-', ' ').replace('.', ' ').upper()
    
    # Cek kecocokan
    score = fuzz.token_set_ratio(clean1, clean2)
    return score >= 85

def find_best_match(peserta_name, certiport_names, threshold=90):
    """
    Mencari kecocokan nama terbaik dengan aturan RELAXED v2:
    - Jika Certiport ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)
    - Gunakan persentase 50% dari kata yang lebih sedikit
    - Minimal selalu 2 kata cocok (kecuali single word dengan - atau .)
    - Threshold dinaikkan ke 90 untuk menghindari false positive
    """
    peserta_normalized = normalize_name(peserta_name)
    peserta_words = [w for w in peserta_normalized.split() if len(w) >= 2]
    peserta_word_count = len(peserta_words)
    
    # Cek exact match
    if peserta_normalized in certiport_names:
        return peserta_normalized, 100, "Exact Match"
    
    # Cek nama terbalik
    reversed_name_str = reverse_name(peserta_normalized)
    if reversed_name_str in certiport_names:
        return reversed_name_str, 100, "Exact Match (Nama Terbalik)"
    
    best_score = 0
    best_match = None
    match_type = ""
    best_word_count = 0
    best_min_required = 0
    
    for cert_name in certiport_names:
        cert_words = [w for w in cert_name.split() if len(w) >= 2]
        cert_word_count = len(cert_words)
        
        # Special handling untuk nama 1 kata
        if peserta_word_count == 1:
            if is_single_word_match(peserta_normalized, cert_name):
                return cert_name, 90, "Single Word Match (dengan tanda - atau .)"
            continue
        
        matching_words, _, _ = count_matching_words(peserta_normalized, cert_name)
        
        # Hitung minimal kata yang harus cocok (RELAXED)
        min_words_required = get_min_matching_words(peserta_word_count, cert_word_count)
        
        # Cek minimum kata yang harus cocok
        if matching_words < min_words_required:
            continue
        
        # Berbagai metode fuzzy matching
        score1 = fuzz.ratio(peserta_normalized, cert_name)
        score2 = fuzz.token_set_ratio(peserta_normalized, cert_name)
        score3 = fuzz.token_sort_ratio(peserta_normalized, cert_name)
        score4 = fuzz.partial_ratio(peserta_normalized, cert_name)
        
        max_score = max(score1, score2, score3, score4)
        
        if matching_words > best_word_count or (matching_words == best_word_count and max_score > best_score):
            best_score = max_score
            best_match = cert_name
            best_word_count = matching_words
            best_min_required = min_words_required
            if max_score == score1:
                match_type = "Fuzzy Ratio"
            elif max_score == score2:
                match_type = "Token Set Ratio"
            elif max_score == score3:
                match_type = "Token Sort Ratio"
            else:
                match_type = "Partial Ratio"
    
    if best_score >= threshold and best_match:
        return best_match, best_score, f"{match_type} ({best_word_count}/{best_min_required} kata cocok)"
    
    return None, best_score, "Tidak Ditemukan"

print("✅ Fungsi matching (IMPROVED v2 - RELAXED) berhasil dibuat!")
print("\n📋 Aturan Matching BARU:")
print("   • Jika Certiport ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)")
print("   • Gunakan persentase 50% dari kata yang LEBIH SEDIKIT")
print("   • Minimal selalu 2 kata cocok")
print("   • Nama 1 kata → harus ada tanda - atau . dan match")
print("\n📌 Contoh Aplikasi:")
print("   Form: 4 kata, Certiport: 2 kata → minimal 2 kata cocok ✓")
print("   Form: 4 kata, Certiport: 4 kata → minimal 2 kata cocok (50%)")
print("   Form: 3 kata, Certiport: 3 kata → minimal 2 kata cocok (50%)")

✅ Fungsi matching (IMPROVED v2 - RELAXED) berhasil dibuat!

📋 Aturan Matching BARU:
   • Jika Certiport ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)
   • Gunakan persentase 50% dari kata yang LEBIH SEDIKIT
   • Minimal selalu 2 kata cocok
   • Nama 1 kata → harus ada tanda - atau . dan match

📌 Contoh Aplikasi:
   Form: 4 kata, Certiport: 2 kata → minimal 2 kata cocok ✓
   Form: 4 kata, Certiport: 4 kata → minimal 2 kata cocok (50%)
   Form: 3 kata, Certiport: 3 kata → minimal 2 kata cocok (50%)


## 5️⃣ Proses Cross-Check dengan Certiport

In [152]:
# ============================================================
# PROSES CROSS-CHECK FORM DENGAN CERTIPORT
# ============================================================
print(f"{'='*70}")
print("🔍 PROSES CROSS-CHECK FORM KEIKUTSERTAAN DENGAN CERTIPORT")
print(f"{'='*70}")

# Buat Full Name dan normalisasi untuk Certiport
certiport_mcf['Full Name'] = certiport_mcf['First Name'].fillna('') + ' ' + certiport_mcf['Last Name'].fillna('')
certiport_mos['Full Name'] = certiport_mos['First Name'].fillna('') + ' ' + certiport_mos['Last Name'].fillna('')

certiport_mcf_names = [normalize_name(name) for name in certiport_mcf['Full Name'].tolist()]
certiport_mos_names = [normalize_name(name) for name in certiport_mos['Full Name'].tolist()]

# Proses matching
form_results = []

print(f"\n⏳ Memproses {len(form_df)} peserta...")

for idx, row in form_df.iterrows():
    nama = row['NAMA LENGKAP']
    nim = row['NIM (Nomor Induk Mahasiswa)']
    no_hp = str(row.get('NOMOR HANDHONE', '')) if pd.notna(row.get('NOMOR HANDHONE')) else '-'
    email = str(row['Email']) if pd.notna(row['Email']) else '-'
    email2 = str(row['Email2']) if pd.notna(row['Email2']) else '-'
    email_final = row['Email Final']
    prodi = row.get('PROGRAM STUDI', '-')
    
    # Ambil subprogram yang di-REQUEST dari form
    subprog_mos = str(row.get('PILIH SUBPROGRAM MOS', '')) if pd.notna(row.get('PILIH SUBPROGRAM MOS')) else ''
    subprog_mcf = str(row.get('PILIH SUBPROGRAM MCF', '')) if pd.notna(row.get('PILIH SUBPROGRAM MCF')) else ''
    subprog_both = str(row.get('PILIH SUBPROGRAM', '')) if pd.notna(row.get('PILIH SUBPROGRAM')) else ''
    
    # Tentukan program yang di-REQUEST
    request_mos = bool(subprog_mos and subprog_mos.strip() and subprog_mos.lower() != 'nan')
    request_mcf = bool(subprog_mcf and subprog_mcf.strip() and subprog_mcf.lower() != 'nan')
    request_both = bool(subprog_both and subprog_both.strip() and subprog_both.lower() != 'nan')
    
    # Jika request_both berisi MOS & MCF sekaligus
    if request_both and ('MOS' in subprog_both.upper() or 'Office' in subprog_both):
        request_mos = True
    if request_both and ('MCF' in subprog_both.upper() or 'Azure' in subprog_both):
        request_mcf = True
    
    # Gabungkan subprogram yang di-request untuk ditampilkan
    subprogram_request = []
    if subprog_mos and subprog_mos.strip() and subprog_mos.lower() != 'nan':
        subprogram_request.append(subprog_mos.strip())
    if subprog_mcf and subprog_mcf.strip() and subprog_mcf.lower() != 'nan':
        subprogram_request.append(subprog_mcf.strip())
    if subprog_both and subprog_both.strip() and subprog_both.lower() != 'nan':
        subprogram_request.append(subprog_both.strip())
    subprogram_display = ' & '.join(subprogram_request) if subprogram_request else '-'
    
    # Tentukan jenis request
    if request_mos and request_mcf:
        jenis_request = 'MOS & MCF'
    elif request_mos:
        jenis_request = 'MOS'
    elif request_mcf:
        jenis_request = 'MCF'
    else:
        jenis_request = 'Tidak Jelas'
    
    # CEK DI CERTIPORT SESUAI REQUEST
    match_mos = None
    match_mcf = None
    score_mos = 0
    score_mcf = 0
    
    if request_mos:
        match_mos, score_mos, match_type_mos = find_best_match(nama, certiport_mos_names)
    if request_mcf:
        match_mcf, score_mcf, match_type_mcf = find_best_match(nama, certiport_mcf_names)
    
    # Tentukan status dan keterangan berdasarkan REQUEST dan hasil CEK
    if request_mos and request_mcf:
        # Request keduanya - cek apakah sudah lulus keduanya
        if match_mos and match_mcf:
            status = '✅ BERHAK SERTIFIKAT KEIKUTSERTAAN'
            keterangan = 'Sudah lulus MOS & MCF → Berhak dapat Sertifikat Keikutsertaan'
            nama_certiport = f"MOS: {match_mos} | MCF: {match_mcf}"
        elif match_mos and not match_mcf:
            status = '⚠️ SEBAGIAN LULUS'
            keterangan = 'Sudah lulus MOS, tapi BELUM lulus MCF → Silakan registrasi MCF'
            nama_certiport = f"MOS: {match_mos}"
        elif match_mcf and not match_mos:
            status = '⚠️ SEBAGIAN LULUS'
            keterangan = 'Sudah lulus MCF, tapi BELUM lulus MOS → Silakan registrasi MOS'
            nama_certiport = f"MCF: {match_mcf}"
        else:
            status = '❌ BELUM SERTIFIKASI'
            keterangan = 'Belum lulus MOS & MCF → Silakan registrasi sertifikasi terlebih dahulu'
            nama_certiport = '-'
    elif request_mos:
        # Request MOS saja
        if match_mos:
            status = '✅ BERHAK SERTIFIKAT KEIKUTSERTAAN'
            keterangan = 'Sudah lulus MOS → Berhak dapat Sertifikat Keikutsertaan'
            nama_certiport = match_mos
        else:
            status = '❌ BELUM SERTIFIKASI'
            keterangan = 'Belum lulus MOS → Silakan registrasi sertifikasi MOS terlebih dahulu'
            nama_certiport = '-'
    elif request_mcf:
        # Request MCF saja
        if match_mcf:
            status = '✅ BERHAK SERTIFIKAT KEIKUTSERTAAN'
            keterangan = 'Sudah lulus MCF → Berhak dapat Sertifikat Keikutsertaan'
            nama_certiport = match_mcf
        else:
            status = '❌ BELUM SERTIFIKASI'
            keterangan = 'Belum lulus MCF → Silakan registrasi sertifikasi MCF terlebih dahulu'
            nama_certiport = '-'
    else:
        status = '⚠️ REQUEST TIDAK JELAS'
        keterangan = 'Tidak ada subprogram yang dipilih di form'
        nama_certiport = '-'
    
    form_results.append({
        'Nama Lengkap': nama,
        'NIM': nim,
        'No HP': no_hp,
        'Email': email,
        'Email2': email2,
        'Email Final': email_final,
        'Program Studi': prodi,
        'Jenis Request': jenis_request,
        'Subprogram Request': subprogram_display,
        'Nama di Certiport': nama_certiport,
        'Skor Kecocokan': max(score_mos, score_mcf),
        'Status': status,
        'Keterangan': keterangan
    })

# Buat DataFrame hasil
form_results_df = pd.DataFrame(form_results)

# ============================================================
# DETEKSI NAMA DOUBLE (DUPLIKAT)
# ============================================================
# Cari nama yang sama persis (case insensitive)
form_results_df['Nama Upper'] = form_results_df['Nama Lengkap'].str.strip().str.upper()
duplicate_names = form_results_df['Nama Upper'].value_counts()
duplicate_names = duplicate_names[duplicate_names > 1].index.tolist()

# Tandai yang double
form_results_df['Is Double'] = form_results_df['Nama Upper'].isin(duplicate_names)
form_results_df = form_results_df.drop(columns=['Nama Upper'])

# Pisahkan hasil berdasarkan status
berhak_sertifikat = form_results_df[form_results_df['Status'].str.contains('BERHAK')].copy()
sebagian_lulus = form_results_df[form_results_df['Status'].str.contains('SEBAGIAN')].copy()
belum_sertifikasi = form_results_df[form_results_df['Status'].str.contains('BELUM')].copy()
request_tidak_jelas = form_results_df[form_results_df['Status'].str.contains('TIDAK JELAS')].copy()

# Hitung nama double
nama_double_count = form_results_df[form_results_df['Is Double']].shape[0]

print(f"\n✅ Proses cross-check selesai!")
print(f"\n{'='*70}")
print("📊 HASIL CROSS-CHECK FORM KEIKUTSERTAAN")
print(f"{'='*70}")
print(f"   ✅ BERHAK SERTIFIKAT KEIKUTSERTAAN : {len(berhak_sertifikat):>4} peserta")
print(f"   ⚠️  SEBAGIAN LULUS                 : {len(sebagian_lulus):>4} peserta")
print(f"   ❌ BELUM SERTIFIKASI               : {len(belum_sertifikasi):>4} peserta")
print(f"   ⚠️  REQUEST TIDAK JELAS            : {len(request_tidak_jelas):>4} peserta")
print(f"   {'─'*45}")
print(f"   TOTAL                              : {len(form_results_df):>4} peserta")
print(f"\n   🟡 NAMA DOUBLE (DUPLIKAT)          : {nama_double_count:>4} peserta")

# Breakdown detail keterangan
print(f"\n📊 Breakdown Detail Keterangan:")
print(form_results_df['Keterangan'].value_counts().to_string())

# Breakdown Jenis Request
print(f"\n📊 Breakdown Jenis Request:")
print(form_results_df['Jenis Request'].value_counts().to_string())

# Tampilkan nama-nama double jika ada
if nama_double_count > 0:
    print(f"\n🟡 DAFTAR NAMA DOUBLE (DUPLIKAT):")
    double_df = form_results_df[form_results_df['Is Double']][['Nama Lengkap', 'NIM', 'Status']].sort_values('Nama Lengkap')
    print(double_df.to_string(index=False))

🔍 PROSES CROSS-CHECK FORM KEIKUTSERTAAN DENGAN CERTIPORT

⏳ Memproses 159 peserta...

✅ Proses cross-check selesai!

📊 HASIL CROSS-CHECK FORM KEIKUTSERTAAN
   ✅ BERHAK SERTIFIKAT KEIKUTSERTAAN :  135 peserta
   ⚠️  SEBAGIAN LULUS                 :    0 peserta
   ❌ BELUM SERTIFIKASI               :   24 peserta
   ⚠️  REQUEST TIDAK JELAS            :    0 peserta
   ─────────────────────────────────────────────
   TOTAL                              :  159 peserta

   🟡 NAMA DOUBLE (DUPLIKAT)          :   33 peserta

📊 Breakdown Detail Keterangan:
Keterangan
Sudah lulus MOS → Berhak dapat Sertifikat Keikutsertaan                 133
Belum lulus MOS → Silakan registrasi sertifikasi MOS terlebih dahulu     22
Belum lulus MCF → Silakan registrasi sertifikasi MCF terlebih dahulu      2
Sudah lulus MOS & MCF → Berhak dapat Sertifikat Keikutsertaan             1
Sudah lulus MCF → Berhak dapat Sertifikat Keikutsertaan                   1

📊 Breakdown Jenis Request:
Jenis Request
MOS          1

## 6️⃣ Tampilkan Detail Hasil

In [153]:
# ============================================================
# TAMPILKAN DETAIL HASIL - BERHAK SERTIFIKAT KEIKUTSERTAAN
# ============================================================
print(f"{'='*90}")
print("✅ PESERTA YANG BERHAK DAPAT SERTIFIKAT KEIKUTSERTAAN")
print(f"{'='*90}")
print(f"Total: {len(berhak_sertifikat)} peserta\n")

if len(berhak_sertifikat) > 0:
    display_cols = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Jenis Request', 'Keterangan', 'Nama di Certiport']
    print(berhak_sertifikat[display_cols].to_string(index=False))
else:
    print("Tidak ada peserta yang berhak dapat sertifikat keikutsertaan")

✅ PESERTA YANG BERHAK DAPAT SERTIFIKAT KEIKUTSERTAAN
Total: 135 peserta

                                Nama Lengkap         NIM          No HP                                           Email Final Jenis Request                                                    Keterangan                                           Nama di Certiport
                          RATU ADISYA FAMELA   202231105 62081363670642                               ratu2231105@itpln.ac.id           MOS       Sudah lulus MOS → Berhak dapat Sertifikat Keikutsertaan                                                 RATU FAMELA
                      MUHAMMAD RAIHAN AZZAKY   202241017    81913054608                             raihan2241017@itpln.ac.id           MOS       Sudah lulus MOS → Berhak dapat Sertifikat Keikutsertaan                                      MUHAMMAD RAIHAN AZZAKY
                           VIA ISNATUL LAILA   202241013  6285600675132                         viaisnatul2241013@itpln.ac.id           MOS  

In [154]:
# ============================================================
# TAMPILKAN DETAIL HASIL - BELUM SERTIFIKASI
# ============================================================
print(f"{'='*90}")
print("❌ PESERTA YANG BELUM SERTIFIKASI (SILAKAN REGISTRASI DULU)")
print(f"{'='*90}")
print(f"Total: {len(belum_sertifikasi)} peserta\n")

if len(belum_sertifikasi) > 0:
    display_cols = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Jenis Request', 'Subprogram Request', 'Keterangan']
    print(belum_sertifikasi[display_cols].to_string(index=False))
else:
    print("Tidak ada peserta yang belum sertifikasi")

# Tampilkan juga yang SEBAGIAN LULUS jika ada
if len(sebagian_lulus) > 0:
    print(f"\n{'='*90}")
    print("⚠️ PESERTA YANG SEBAGIAN LULUS (PERLU REGISTRASI SATU LAGI)")
    print(f"{'='*90}")
    print(f"Total: {len(sebagian_lulus)} peserta\n")
    display_cols = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Jenis Request', 'Subprogram Request', 'Keterangan']
    print(sebagian_lulus[display_cols].to_string(index=False))

❌ PESERTA YANG BELUM SERTIFIKASI (SILAKAN REGISTRASI DULU)
Total: 24 peserta

                   Nama Lengkap       NIM         No HP                                       Email Final Jenis Request    Subprogram Request                                                           Keterangan
             JAPAR SIRINGORINGO 202214045   85212078991                          japar2214045@itpln.ac.id           MOS MOS: Office Word 2019 Belum lulus MOS → Silakan registrasi sertifikasi MOS terlebih dahulu
             JAPAR SIRINGORINGO 202214045   85212078991                          japar2214045@itpln.ac.id           MOS MOS: Office Word 2019 Belum lulus MOS → Silakan registrasi sertifikasi MOS terlebih dahulu
     LALU MUHAMMAD RISGAN NAZWA 202231009 6285973913275                           Lalu2231009@itpln.ac.id           MOS MOS: Office Word 2019 Belum lulus MOS → Silakan registrasi sertifikasi MOS terlebih dahulu
        ANNA FINCE MARIANA WOUW 202111062  852116359972                       

## 7️⃣ Dashboard Summary

In [155]:
# ============================================================
# DASHBOARD SUMMARY
# ============================================================
print(f"\n{'='*90}")
print(" " * 30 + "📊 DASHBOARD SUMMARY")
print(f"{'='*90}")

total_form = len(form_df)
mos_request = len(form_results_df[form_results_df['Jenis Request'] == 'MOS'])
mcf_request = len(form_results_df[form_results_df['Jenis Request'] == 'MCF'])
both_request = len(form_results_df[form_results_df['Jenis Request'] == 'MOS & MCF'])

print(f"""
┌────────────────────────────────────────────────────────────────────────────────┐
│                    DATA FORM KEIKUTSERTAAN SERTIFIKASI                         │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Peserta Form           : {total_form:>5} peserta                                 │
│    ├─ Request MOS saja        : {mos_request:>5} peserta ({mos_request/total_form*100 if total_form > 0 else 0:>5.1f}%)                  │
│    ├─ Request MCF saja        : {mcf_request:>5} peserta ({mcf_request/total_form*100 if total_form > 0 else 0:>5.1f}%)                  │
│    └─ Request MOS & MCF       : {both_request:>5} peserta ({both_request/total_form*100 if total_form > 0 else 0:>5.1f}%)                  │
├────────────────────────────────────────────────────────────────────────────────┤
│                          DATA CERTIPORT                                        │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Record Certiport       : {len(certiport_df):>5} records                                │
│    ├─ MCF (AI-900)            : {len(certiport_mcf):>5} records                                │
│    └─ MOS (Office 2019)       : {len(certiport_mos):>5} records                                │
├────────────────────────────────────────────────────────────────────────────────┤
│                       HASIL VERIFIKASI STATUS                                  │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ BERHAK SERTIFIKAT         : {len(berhak_sertifikat):>5} peserta ({len(berhak_sertifikat)/total_form*100 if total_form > 0 else 0:>5.1f}%)                  │
│  ⚠️  SEBAGIAN LULUS           : {len(sebagian_lulus):>5} peserta ({len(sebagian_lulus)/total_form*100 if total_form > 0 else 0:>5.1f}%)                  │
│  ❌ BELUM SERTIFIKASI         : {len(belum_sertifikasi):>5} peserta ({len(belum_sertifikasi)/total_form*100 if total_form > 0 else 0:>5.1f}%)                  │
├────────────────────────────────────────────────────────────────────────────────┤
│  🟡 NAMA DOUBLE (DUPLIKAT)    : {nama_double_count:>5} peserta                                 │
└────────────────────────────────────────────────────────────────────────────────┘
""")

print(f"📌 KETERANGAN STATUS:")
print(f"   ✅ BERHAK SERTIFIKAT    = Sudah lulus sesuai yang di-REQUEST, berhak dapat Sertifikat Keikutsertaan")
print(f"   ⚠️  SEBAGIAN LULUS      = Request MOS & MCF, tapi baru lulus salah satu")
print(f"   ❌ BELUM SERTIFIKASI    = Belum lulus, silakan registrasi sertifikasi terlebih dahulu")
print(f"   🟡 NAMA DOUBLE          = Nama yang sama persis muncul lebih dari 1x (ditandai kuning di Excel)")


                              📊 DASHBOARD SUMMARY

┌────────────────────────────────────────────────────────────────────────────────┐
│                    DATA FORM KEIKUTSERTAAN SERTIFIKASI                         │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Peserta Form           :   159 peserta                                 │
│    ├─ Request MOS saja        :   155 peserta ( 97.5%)                  │
│    ├─ Request MCF saja        :     3 peserta (  1.9%)                  │
│    └─ Request MOS & MCF       :     1 peserta (  0.6%)                  │
├────────────────────────────────────────────────────────────────────────────────┤
│                          DATA CERTIPORT                                        │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Record Certiport       :  4853 records                                │
│    ├─ MCF (AI-900)            :   897 records            

## 8️⃣ Export Hasil ke CSV

In [156]:
# ============================================================
# EXPORT HASIL KE FILE CSV
# ============================================================
print(f"{'='*70}")
print("💾 EXPORT HASIL KE FILE CSV")
print(f"{'='*70}")

# Kolom untuk export
export_cols = ['Nama Lengkap', 'NIM', 'No HP', 'Email', 'Email2', 'Email Final', 'Program Studi', 
               'Jenis Request', 'Subprogram Request', 'Nama di Certiport', 'Skor Kecocokan', 
               'Status', 'Keterangan', 'Is Double']

# Export peserta yang BERHAK dapat sertifikat keikutsertaan
berhak_sertifikat[export_cols].to_csv('FORM_BERHAK_SERTIFIKAT.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ FORM_BERHAK_SERTIFIKAT.csv")
print(f"   → {len(berhak_sertifikat)} peserta yang BERHAK dapat Sertifikat Keikutsertaan")

# Export peserta yang BELUM sertifikasi
belum_sertifikasi[export_cols].to_csv('FORM_BELUM_SERTIFIKASI.csv', index=False, encoding='utf-8-sig')
print(f"\n❌ FORM_BELUM_SERTIFIKASI.csv")
print(f"   → {len(belum_sertifikasi)} peserta yang BELUM SERTIFIKASI (silakan registrasi dulu)")

# Export peserta yang SEBAGIAN LULUS jika ada
if len(sebagian_lulus) > 0:
    sebagian_lulus[export_cols].to_csv('FORM_SEBAGIAN_LULUS.csv', index=False, encoding='utf-8-sig')
    print(f"\n⚠️ FORM_SEBAGIAN_LULUS.csv")
    print(f"   → {len(sebagian_lulus)} peserta yang SEBAGIAN LULUS (perlu registrasi satu lagi)")

# Export semua hasil
form_results_df.to_csv('FORM_SEMUA_HASIL.csv', index=False, encoding='utf-8-sig')
print(f"\n📊 FORM_SEMUA_HASIL.csv")
print(f"   → Semua hasil cross-check ({len(form_results_df)} peserta)")

💾 EXPORT HASIL KE FILE CSV

✅ FORM_BERHAK_SERTIFIKAT.csv
   → 135 peserta yang BERHAK dapat Sertifikat Keikutsertaan

❌ FORM_BELUM_SERTIFIKASI.csv
   → 24 peserta yang BELUM SERTIFIKASI (silakan registrasi dulu)

📊 FORM_SEMUA_HASIL.csv
   → Semua hasil cross-check (159 peserta)


## 9️⃣ Export ke Excel dengan Multiple Sheets

In [157]:
# ============================================================
# EXPORT KE EXCEL DENGAN FORMATTING
# ============================================================
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print(f"{'='*70}")
print("📊 EXPORT KE EXCEL DENGAN MULTIPLE SHEETS")
print(f"{'='*70}")

wb = Workbook()

# Style definitions
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
berhak_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # Hijau
belum_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")   # Merah
sebagian_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid") # Orange muda
double_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")   # Kuning untuk DOUBLE
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_worksheet(ws):
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin_border
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.border = thin_border
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 50)

# Sheet 1: Dashboard Summary
ws1 = wb.active
ws1.title = "Dashboard Summary"
total_form = len(form_results_df)
summary = [
    ["KATEGORI", "JUMLAH", "PERSENTASE", "STATUS"],
    ["Total Peserta Form", total_form, "100%", "ℹ️"],
    ["  ├─ Request MOS saja", len(form_results_df[form_results_df['Jenis Request'] == 'MOS']), 
     f"{len(form_results_df[form_results_df['Jenis Request'] == 'MOS'])/total_form*100:.1f}%" if total_form > 0 else "0%", ""],
    ["  ├─ Request MCF saja", len(form_results_df[form_results_df['Jenis Request'] == 'MCF']), 
     f"{len(form_results_df[form_results_df['Jenis Request'] == 'MCF'])/total_form*100:.1f}%" if total_form > 0 else "0%", ""],
    ["  └─ Request MOS & MCF", len(form_results_df[form_results_df['Jenis Request'] == 'MOS & MCF']), 
     f"{len(form_results_df[form_results_df['Jenis Request'] == 'MOS & MCF'])/total_form*100:.1f}%" if total_form > 0 else "0%", ""],
    ["", "", "", ""],
    ["BERHAK SERTIFIKAT", len(berhak_sertifikat), f"{len(berhak_sertifikat)/total_form*100:.1f}%" if total_form > 0 else "0%", "✅ Sudah lulus sesuai request"],
    ["SEBAGIAN LULUS", len(sebagian_lulus), f"{len(sebagian_lulus)/total_form*100:.1f}%" if total_form > 0 else "0%", "⚠️ Request dua, lulus satu"],
    ["BELUM SERTIFIKASI", len(belum_sertifikasi), f"{len(belum_sertifikasi)/total_form*100:.1f}%" if total_form > 0 else "0%", "❌ Silakan registrasi dulu"],
    ["", "", "", ""],
    ["NAMA DOUBLE (DUPLIKAT)", nama_double_count, "", "🟡 Ditandai kuning di Excel"],
]
for row in summary:
    ws1.append(row)
style_worksheet(ws1)

# Sheet 2: BERHAK SERTIFIKAT KEIKUTSERTAAN
ws2 = wb.create_sheet("Berhak Sertifikat")
cols_berhak = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Program Studi', 'Jenis Request', 'Subprogram Request', 'Keterangan', 'Nama di Certiport', 'Is Double']
if len(berhak_sertifikat) > 0:
    for r in dataframe_to_rows(berhak_sertifikat[cols_berhak], index=False, header=True):
        ws2.append(r)
else:
    ws2.append(cols_berhak)
    ws2.append(["Tidak ada data"] + [""]*(len(cols_berhak)-1))
style_worksheet(ws2)
# Warnai baris - kuning jika DOUBLE, hijau jika tidak
for row_idx, row in enumerate(ws2.iter_rows(min_row=2, max_row=ws2.max_row), start=2):
    is_double = str(ws2.cell(row=row_idx, column=10).value)  # Kolom Is Double
    fill = double_fill if is_double == 'True' else berhak_fill
    for cell in row:
        cell.fill = fill

# Sheet 3: BELUM SERTIFIKASI
ws3 = wb.create_sheet("Belum Sertifikasi")
cols_belum = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Program Studi', 'Jenis Request', 'Subprogram Request', 'Keterangan', 'Is Double']
if len(belum_sertifikasi) > 0:
    for r in dataframe_to_rows(belum_sertifikasi[cols_belum], index=False, header=True):
        ws3.append(r)
else:
    ws3.append(cols_belum)
    ws3.append(["Tidak ada data"] + [""]*(len(cols_belum)-1))
style_worksheet(ws3)
# Warnai baris - kuning jika DOUBLE, merah jika tidak
for row_idx, row in enumerate(ws3.iter_rows(min_row=2, max_row=ws3.max_row), start=2):
    is_double = str(ws3.cell(row=row_idx, column=9).value)  # Kolom Is Double
    fill = double_fill if is_double == 'True' else belum_fill
    for cell in row:
        cell.fill = fill

# Sheet 4: SEBAGIAN LULUS (jika ada)
if len(sebagian_lulus) > 0:
    ws_sebagian = wb.create_sheet("Sebagian Lulus")
    cols_sebagian = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Program Studi', 'Jenis Request', 'Subprogram Request', 'Keterangan', 'Nama di Certiport', 'Is Double']
    for r in dataframe_to_rows(sebagian_lulus[cols_sebagian], index=False, header=True):
        ws_sebagian.append(r)
    style_worksheet(ws_sebagian)
    # Warnai baris - kuning jika DOUBLE, orange jika tidak
    for row_idx, row in enumerate(ws_sebagian.iter_rows(min_row=2, max_row=ws_sebagian.max_row), start=2):
        is_double = str(ws_sebagian.cell(row=row_idx, column=10).value)  # Kolom Is Double
        fill = double_fill if is_double == 'True' else sebagian_fill
        for cell in row:
            cell.fill = fill

# Sheet 5: Semua Hasil
ws4 = wb.create_sheet("Semua Hasil")
cols_all = ['Nama Lengkap', 'NIM', 'No HP', 'Email Final', 'Jenis Request', 'Subprogram Request', 'Keterangan', 'Nama di Certiport', 'Status', 'Is Double']
for r in dataframe_to_rows(form_results_df[cols_all], index=False, header=True):
    ws4.append(r)
style_worksheet(ws4)
# Warnai berdasarkan status dan DOUBLE
for row_idx, row in enumerate(ws4.iter_rows(min_row=2, max_row=ws4.max_row), start=2):
    status = str(ws4.cell(row=row_idx, column=9).value)  # Kolom Status
    is_double = str(ws4.cell(row=row_idx, column=10).value)  # Kolom Is Double
    
    if is_double == 'True':
        fill = double_fill  # Kuning untuk DOUBLE
    elif 'BERHAK' in status:
        fill = berhak_fill  # Hijau
    elif 'SEBAGIAN' in status:
        fill = sebagian_fill  # Orange
    else:
        fill = belum_fill  # Merah
    
    for cell in row:
        cell.fill = fill

# Simpan file (overwrite jika sudah ada)
excel_file = 'HASIL_FORM_KEIKUTSERTAAN.xlsx'
wb.save(excel_file)

print(f"\n✅ File Excel berhasil disimpan: {excel_file}")
print(f"\n📋 DAFTAR SHEETS:")
print(f"   1. Dashboard Summary      - Ringkasan keseluruhan")
print(f"   2. Berhak Sertifikat      - 🟢 {len(berhak_sertifikat)} peserta (BERHAK dapat sertifikat keikutsertaan)")
print(f"   3. Belum Sertifikasi      - 🔴 {len(belum_sertifikasi)} peserta (silakan registrasi dulu)")
if len(sebagian_lulus) > 0:
    print(f"   4. Sebagian Lulus         - 🟠 {len(sebagian_lulus)} peserta (request 2, lulus 1)")
    print(f"   5. Semua Hasil            - Detail lengkap semua peserta")
else:
    print(f"   4. Semua Hasil            - Detail lengkap semua peserta")
print(f"\n📌 KETERANGAN WARNA:")
print(f"   🟢 Hijau  = Berhak dapat Sertifikat Keikutsertaan")
print(f"   🟠 Orange = Sebagian Lulus (perlu registrasi satu lagi)")
print(f"   🔴 Merah  = Belum Sertifikasi (silakan registrasi dulu)")
print(f"   🟡 Kuning = NAMA DOUBLE (duplikat) - perlu dicek manual")

📊 EXPORT KE EXCEL DENGAN MULTIPLE SHEETS

✅ File Excel berhasil disimpan: HASIL_FORM_KEIKUTSERTAAN.xlsx

📋 DAFTAR SHEETS:
   1. Dashboard Summary      - Ringkasan keseluruhan
   2. Berhak Sertifikat      - 🟢 135 peserta (BERHAK dapat sertifikat keikutsertaan)
   3. Belum Sertifikasi      - 🔴 24 peserta (silakan registrasi dulu)
   4. Semua Hasil            - Detail lengkap semua peserta

📌 KETERANGAN WARNA:
   🟢 Hijau  = Berhak dapat Sertifikat Keikutsertaan
   🟠 Orange = Sebagian Lulus (perlu registrasi satu lagi)
   🔴 Merah  = Belum Sertifikasi (silakan registrasi dulu)
   🟡 Kuning = NAMA DOUBLE (duplikat) - perlu dicek manual


## 🔍 Fungsi Pencarian Manual

In [158]:
# ============================================================
# FUNGSI PENCARIAN MANUAL
# ============================================================

def cari_nama(nama):
    """
    Fungsi untuk mencari nama secara manual di database Certiport
    Akan melakukan pengecekan di KEDUA database (MOS dan MCF)
    
    Parameter:
    - nama: nama yang ingin dicari
    
    Contoh: cari_nama('JOHN DOE')
    """
    match_mos, score_mos, match_type_mos = find_best_match(nama, certiport_mos_names)
    match_mcf, score_mcf, match_type_mcf = find_best_match(nama, certiport_mcf_names)
    
    print(f"\n🔍 Hasil Pencarian untuk: {nama}")
    print(f"   {'='*70}")
    
    print(f"\n   📋 HASIL CEK MOS (Office 2019):")
    if match_mos:
        print(f"      ✅ DITEMUKAN")
        print(f"      Nama di Certiport: {match_mos}")
        print(f"      Skor Kecocokan: {score_mos}")
        print(f"      Tipe Match: {match_type_mos}")
    else:
        print(f"      ❌ TIDAK DITEMUKAN")
        print(f"      Skor Tertinggi: {score_mos}")
    
    print(f"\n   📋 HASIL CEK MCF (Azure AI-900):")
    if match_mcf:
        print(f"      ✅ DITEMUKAN")
        print(f"      Nama di Certiport: {match_mcf}")
        print(f"      Skor Kecocokan: {score_mcf}")
        print(f"      Tipe Match: {match_type_mcf}")
    else:
        print(f"      ❌ TIDAK DITEMUKAN")
        print(f"      Skor Tertinggi: {score_mcf}")
    
    print(f"\n   {'='*70}")
    print(f"   📊 KESIMPULAN:")
    if match_mos and match_mcf:
        print(f"      ✅ Sudah sertifikasi MOS & MCF (keduanya)")
    elif match_mos and not match_mcf:
        print(f"      ✅ Sudah sertifikasi MOS saja (belum MCF)")
    elif match_mcf and not match_mos:
        print(f"      ✅ Sudah sertifikasi MCF saja (belum MOS)")
    else:
        print(f"      ❌ Belum pernah sertifikasi MOS maupun MCF")

print("✅ Fungsi cari_nama() siap digunakan!")
print("\n📖 Cara Pakai:")
print("   cari_nama('NAMA LENGKAP')  # Akan cek di MOS dan MCF sekaligus")

✅ Fungsi cari_nama() siap digunakan!

📖 Cara Pakai:
   cari_nama('NAMA LENGKAP')  # Akan cek di MOS dan MCF sekaligus


In [159]:
# ============================================================
# CONTOH PENGGUNAAN FUNGSI PENCARIAN
# ============================================================

# Contoh: cari_nama('MUHAMMAD REVIANSYAH')

In [160]:
# ============================================================
# DEBUG: CEK KASUS RITCHIE CHORINUS TIOLUNG MAABUAT (DENGAN ATURAN BARU)
# ============================================================
print("="*80)
print("🔍 DEBUG: ANALISIS KASUS RITCHIE DENGAN ATURAN RELAXED v2")
print("="*80)

# Nama dari form
nama_form = "RITCHIE CHORINUS TIOLUNG MAABUAT"
nama_normalized = normalize_name(nama_form)
peserta_words = [w for w in nama_normalized.split() if len(w) >= 2]

print(f"\n📋 Nama di Form: {nama_form}")
print(f"   Normalized: {nama_normalized}")
print(f"   Jumlah kata: {len(peserta_words)} kata")

# Simulasi dengan nama Certiport yang pendek
nama_certiport = "RITCHIE MAABUAT"
cert_words = [w for w in nama_certiport.split() if len(w) >= 2]
matching, _, _ = count_matching_words(nama_normalized, nama_certiport)
min_required = get_min_matching_words(len(peserta_words), len(cert_words))

print(f"\n📊 ANALISIS DENGAN ATURAN BARU:")
print(f"   Nama di Certiport: {nama_certiport}")
print(f"   Jumlah kata Certiport: {len(cert_words)} kata")
print(f"   Kata yang cocok: {matching} (RITCHIE, MAABUAT)")
print(f"   Minimal diperlukan: {min_required} kata")
print(f"   Hasil: {'✅ COCOK' if matching >= min_required else '❌ TIDAK COCOK'}")

# Cari di Certiport dengan fungsi baru
print(f"\n🔍 Pencarian di Certiport MOS (dengan aturan RELAXED):")
match_mos, score_mos, match_type_mos = find_best_match(nama_form, certiport_mos_names)

if match_mos:
    print(f"   ✅ DITEMUKAN: {match_mos}")
    print(f"   Skor: {score_mos}")
    print(f"   Tipe: {match_type_mos}")
else:
    print(f"   ❌ TIDAK DITEMUKAN")
    print(f"   Skor tertinggi: {score_mos}")

print(f"\n💡 PENJELASAN ATURAN RELAXED v2:")
print(f"   • Certiport hanya punya 2 kata (RITCHIE MAABUAT)")
print(f"   • Karena Certiport ≤2 kata → minimal 2 kata cocok")
print(f"   • Form punya 4 kata, 2 kata cocok = memenuhi syarat ✓")

🔍 DEBUG: ANALISIS KASUS RITCHIE DENGAN ATURAN RELAXED v2

📋 Nama di Form: RITCHIE CHORINUS TIOLUNG MAABUAT
   Normalized: RITCHIE CHORINUS TIOLUNG MAABUAT
   Jumlah kata: 4 kata

📊 ANALISIS DENGAN ATURAN BARU:
   Nama di Certiport: RITCHIE MAABUAT
   Jumlah kata Certiport: 2 kata
   Kata yang cocok: 2 (RITCHIE, MAABUAT)
   Minimal diperlukan: 2 kata
   Hasil: ✅ COCOK

🔍 Pencarian di Certiport MOS (dengan aturan RELAXED):
   ✅ DITEMUKAN: RITCHIE MAABUAT
   Skor: 100
   Tipe: Token Set Ratio (2/2 kata cocok)

💡 PENJELASAN ATURAN RELAXED v2:
   • Certiport hanya punya 2 kata (RITCHIE MAABUAT)
   • Karena Certiport ≤2 kata → minimal 2 kata cocok
   • Form punya 4 kata, 2 kata cocok = memenuhi syarat ✓


---
## 📌 Ringkasan Output Files

| File | Deskripsi |
|------|-----------|
| `FORM_SUDAH_SERTIFIKASI.csv` | Peserta yang SUDAH PERNAH sertifikasi (ada di Certiport) |
| `FORM_BELUM_SERTIFIKASI.csv` | Peserta yang BELUM PERNAH sertifikasi (tidak di Certiport) |
| `FORM_SEMUA_HASIL.csv` | Semua hasil cross-check |
| `HASIL_FORM_KEIKUTSERTAAN.xlsx` | File Excel dengan multiple sheets dan formatting |

---